# Heterogeneous Blanchard OLG — Chebyshev Spectral PINN

## Master's Thesis — Cecilia Trojani (UZH/ETH, 2025-2026)

---

This notebook replaces the MLP correction networks in the
structural-baseline PINN with **finite Chebyshev polynomial series**:
$$N^i(f) = \sum_{k=0}^{K} a^i_k \, T_k(2f - 1)$$

The trial solution is unchanged:
$$\widehat\phi^i(f) = \phi^i_{\text{base}}(f) \cdot \exp\big(\epsilon^i(f) \cdot N^i(f)\big)$$

Only the parameterization of $N^i$ changes.

### Why spectral?

For smooth solutions on a fixed interval, an MLP with ~9000 parameters
is overkill. Chebyshev series have:

- **Exponential convergence** for smooth functions (vs polynomial for MLPs)
- **Analytical derivatives** via the Chebyshev recurrence (better numerical conditioning)
- **Much smaller parameter count**: $4 \times (K+1)$ coefficients total
- **Better-behaved loss landscape**: convex quadratic in coefficients (for linear problems; nearly so for ours)

For $K=15$: 64 trainable coefficients total, vs ~36000 for the MLP version.

### Validation strategy (same as MLP version)

Three sequential runs:
1. $\gamma^1 = \gamma^2 = 1$ (homog log) — should recover constants
2. $\gamma^1 = \gamma^2 = 2$ (homog CRRA) — should recover constants
3. $\gamma^1 = 1, \gamma^2 = 2$ (truly heterogeneous) — production result

Expected outcome: cleaner residual floor in Run 3, possibly faster
convergence due to smaller, better-conditioned parameter space.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

torch.set_default_dtype(torch.float64)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}, dtype: {torch.get_default_dtype()}')

# ── Common primitives (same as structural-baseline notebook) ──────────────
rho   = 0.05
nu    = 0.02
mu_Y  = 0.02
sig_Y = 0.10
omega = 0.70
sig_S = sig_Y

ALPHA_POP1 = 0.5
ALPHA_POP2 = 0.5

F_EPS = 1e-3

def beta_crra(g):
    A = rho + (g-1)*mu_Y - 0.5*g*(g-1)*sig_Y**2 + nu*(1 + g + omega*(g-1))
    B = rho + (g-1)*mu_Y - 0.5*g*(g-1)*sig_Y**2 + g*nu
    disc = A**2 - 4*g*nu*omega*B
    return (A - math.sqrt(disc)) / (2*g*nu)

def homog_constants(g):
    b = beta_crra(g)
    r = rho + g*mu_Y - 0.5*g*(g+1)*sig_Y**2 + g*nu*(1-b)
    phi   = 1.0/(nu + rho/g + (g-1)/g*r + 0.5*(g-1)*sig_Y**2)
    phi_e = omega    /(r + nu - mu_Y + g*sig_Y**2)
    phi_d = (1-omega)/(r       - mu_Y + g*sig_Y**2)
    return b, r, phi, phi_e, phi_d


---
## 1. Chebyshev polynomial module

`ChebyshevSeries(K)` evaluates $N(f) = \sum_{k=0}^{K} a_k T_k(2f-1)$
on a batch of f-values. We use the three-term recurrence
$T_{k+1}(x) = 2x\,T_k(x) - T_{k-1}(x)$ with $T_0(x)=1$, $T_1(x)=x$.
This is numerically stable and differentiable through autograd, so the
residual function gets clean second derivatives.

The coefficients are initialized near zero so the network starts at the
structural baseline (no correction).


In [ ]:
class ChebyshevSeries(nn.Module):
    '''Finite Chebyshev polynomial series N(f) = sum_{k=0}^K a_k T_k(2f-1).
    f is expected in [0, 1]; we map to x = 2f-1 in [-1, 1].'''
    def __init__(self, K=15, init_scale=1e-3, seed=None):
        super().__init__()
        self.K = K
        if seed is not None:
            torch.manual_seed(seed)
        # Small random init so different N's break symmetry
        self.coeffs = nn.Parameter(init_scale * torch.randn(K + 1))

    def forward(self, f):
        # Map f in [0,1] to x in [-1,1]
        x = 2.0 * f - 1.0
        # Chebyshev recurrence
        T_prev = torch.ones_like(x)
        result = self.coeffs[0] * T_prev
        if self.K == 0:
            return result
        T_curr = x
        result = result + self.coeffs[1] * T_curr
        for k in range(2, self.K + 1):
            T_next = 2.0 * x * T_curr - T_prev
            result = result + self.coeffs[k] * T_next
            T_prev = T_curr
            T_curr = T_next
        return result


def setup_run(gamma1, gamma2, K=15, seed=42):
    '''Build a spectral het-PINN for given (gamma1, gamma2).
    Returns (net, compute_residuals, constants_dict).'''
    torch.manual_seed(seed)
    np.random.seed(seed)

    # ── Constants ────────────────────────────────────────────────────────
    alpha1_pref = 1.0 - 1.0/gamma1
    alpha2_pref = 1.0 - 1.0/gamma2
    b1_h = beta_crra(gamma1)
    b2_h = beta_crra(gamma2)

    _, _, PHI1_INF, PHIE_R, PHID_R = homog_constants(gamma1)
    _, _, PHI2_INF, PHIE_L, PHID_L = homog_constants(gamma2)

    # ── Equilibrium quantity helpers ──────────────────────────────────────
    def R_of_f(f):    return 1.0 / (f/gamma1 + (1-f)/gamma2)
    def P_of_f(f):
        num = f/gamma1**2 + (1-f)/gamma2**2
        den = (f/gamma1 + (1-f)/gamma2)**2
        return num/den + R_of_f(f)
    def theta_of_f(f):  return R_of_f(f) * sig_Y
    def sigma_f_of_f(f): return f * (R_of_f(f)/gamma1 - 1.0) * sig_Y
    def beta_base(f):    return (1.0 - f)*b2_h + f*b1_h
    def r_base(f):
        R = R_of_f(f); P = P_of_f(f)
        return rho + R*mu_Y - 0.5*R*P*sig_Y**2 + R*nu*(1.0 - beta_base(f))

    # ── Structural baselines ─────────────────────────────────────────────
    def phi1_base(f):
        r = r_base(f)
        return 1.0/(nu + rho/gamma1 + (gamma1-1.0)/gamma1*r + 0.5*(gamma1-1.0)*sig_Y**2)
    def phi2_base(f):
        r = r_base(f)
        return 1.0/(nu + rho/gamma2 + (gamma2-1.0)/gamma2*r + 0.5*(gamma2-1.0)*sig_Y**2)
    def phie_base(f):
        r = r_base(f); R = R_of_f(f)
        return omega/(r + nu - mu_Y + R*sig_Y**2)
    def phid_base(f):
        r = r_base(f); R = R_of_f(f)
        return (1.0 - omega)/(r - mu_Y + R*sig_Y**2)

    # ── Network: 4 independent Chebyshev series ───────────────────────────
    class HardBC_Het_Spectral(nn.Module):
        def __init__(self):
            super().__init__()
            self.N1 = ChebyshevSeries(K, seed=seed+1)
            self.N2 = ChebyshevSeries(K, seed=seed+2)
            self.Ne = ChebyshevSeries(K, seed=seed+3)
            self.Nd = ChebyshevSeries(K, seed=seed+4)

        def forward(self, f):
            n1 = self.N1(f); n2 = self.N2(f); ne = self.Ne(f); nd = self.Nd(f)
            phi1 = phi1_base(f) * torch.exp((1.0 - f) * n1)
            phi2 = phi2_base(f) * torch.exp(f * n2)
            phie = phie_base(f) * torch.exp(f * (1.0 - f) * ne)
            phid = phid_base(f) * torch.exp(f * (1.0 - f) * nd)
            return phi1, phi2, phie, phid

    net = HardBC_Het_Spectral().to(device)
    n_params = sum(p.numel() for p in net.parameters())
    print(f'  Network: 4 × ChebyshevSeries(K={K}) → {n_params} parameters total')

    # ── Residual function (endogenous β) ──────────────────────────────────
    def compute_residuals(net, f):
        phi1, phi2, phie, phid = net(f)
        g_one = torch.ones_like(phi1)
        dphi1, = torch.autograd.grad(phi1, f, grad_outputs=g_one, create_graph=True)
        dphi2, = torch.autograd.grad(phi2, f, grad_outputs=g_one, create_graph=True)
        dphie, = torch.autograd.grad(phie, f, grad_outputs=g_one, create_graph=True)
        dphid, = torch.autograd.grad(phid, f, grad_outputs=g_one, create_graph=True)
        d2phi1, = torch.autograd.grad(dphi1, f, grad_outputs=g_one, create_graph=True)
        d2phi2, = torch.autograd.grad(dphi2, f, grad_outputs=g_one, create_graph=True)
        d2phie, = torch.autograd.grad(dphie, f, grad_outputs=g_one, create_graph=True)
        d2phid, = torch.autograd.grad(dphid, f, grad_outputs=g_one, create_graph=True)

        b1 = ALPHA_POP1 * phie / phi1
        b2 = ALPHA_POP2 * phie / phi2
        b = b1 + b2

        R = R_of_f(f); P = P_of_f(f); th = theta_of_f(f)
        r = rho + R*mu_Y - 0.5*R*P*sig_Y**2 + R*nu*(1.0 - b)
        sf = sigma_f_of_f(f); half_sf2 = 0.5 * sf**2

        Rg = R / gamma1
        theta_bracket = ((Rg - 1.0)*mu_Y
                       + nu * Rg * (1.0 - b)
                       + (1.0 - 0.5*Rg*P + 0.5*(1.0 + 1.0/gamma1)*R**2/gamma1 - Rg) * sig_Y**2)
        # μ_f formula (Prop 3.3 / eq. 5.12): ν(β¹ − f) + f·[bracket]
        # Note: β¹ already includes the population share α¹_pop via b1 definition.
        muf = nu * (b1 - f) + f * theta_bracket

        def A_of(g):
            return (-nu - rho/g
                    + (1.0 - 1.0/g)*(-r)
                    + 0.5*(1.0 - 1.0/g)*(-1.0/g)*th**2)

        L1 = phi1 * A_of(gamma1) + 1.0 + dphi1 * muf + d2phi1 * half_sf2
        L2 = phi2 * A_of(gamma2) + 1.0 + dphi2 * muf + d2phi2 * half_sf2
        Le = phie * (-nu + mu_Y - r - th*sig_Y) + omega       + dphie * muf + d2phie * half_sf2
        Ld = phid * (       mu_Y - r - th*sig_Y) + (1.0-omega) + dphid * muf + d2phid * half_sf2
        Lmc = f*phi1 + (1.0 - f)*phi2 - phie - phid
        return L1, L2, Le, Ld, Lmc

    constants = {
        'gamma1': gamma1, 'gamma2': gamma2,
        'PHI1_INF': PHI1_INF, 'PHI2_INF': PHI2_INF,
        'PHIE_L': PHIE_L, 'PHIE_R': PHIE_R,
        'PHID_L': PHID_L, 'PHID_R': PHID_R,
        'b1_homog': b1_h, 'b2_homog': b2_h,
        'phi1_base': phi1_base, 'phi2_base': phi2_base,
        'phie_base': phie_base, 'phid_base': phid_base,
        'R_of_f': R_of_f, 'P_of_f': P_of_f, 'theta_of_f': theta_of_f,
        'sigma_f_of_f': sigma_f_of_f, 'r_base': r_base, 'beta_base': beta_base,
        'K': K,
    }
    return net, compute_residuals, constants

print('setup_run (spectral) defined.')


---
## 2. Training and validation helpers

`train_run` uses a higher learning rate than the MLP version because
the parameter count is ~500× smaller and the loss landscape is
much better conditioned. Adam still helps initial convergence but
L-BFGS dominates the polish since it can effectively use the full
Hessian on such a small parameter space.


In [ ]:
def sample_f(n):
    n_bdy_l = int(0.20 * n); n_bdy_r = int(0.20 * n)
    n_int   = n - n_bdy_l - n_bdy_r
    f_bdy_l = F_EPS + (0.10 - F_EPS) * torch.rand(n_bdy_l, device=device)
    f_bdy_r = 0.90  + (1.0 - F_EPS - 0.90) * torch.rand(n_bdy_r, device=device)
    f_int   = F_EPS + (1.0 - 2*F_EPS) * torch.rand(n_int, device=device)
    return torch.cat([f_bdy_l, f_int, f_bdy_r]).requires_grad_(True)


def train_run(net, compute_residuals, n_adam=1500, n_lbfgs=100, n_coll=4000,
              lr=1e-3, w_agent=1.0, w_asset=1.0, w_mc=0.5, verbose_every=200):
    '''Adam (warm-up) + L-BFGS (polish) on spectral parameters.'''
    opt = optim.Adam(net.parameters(), lr=lr)
    sched = optim.lr_scheduler.MultiStepLR(opt, milestones=[int(0.5*n_adam), int(0.83*n_adam)], gamma=0.4)

    history = []
    print(f'  Adam: {n_adam} ep, lr={lr}, N_coll={n_coll}, w(agent,asset,mc)=({w_agent},{w_asset},{w_mc})')
    for ep in range(1, n_adam + 1):
        f = sample_f(n_coll)
        L1, L2, Le, Ld, Lmc = compute_residuals(net, f)
        loss = (w_agent*(L1.pow(2).mean() + L2.pow(2).mean())
              + w_asset*(Le.pow(2).mean() + Ld.pow(2).mean())
              + w_mc   * Lmc.pow(2).mean())
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=10.0)
        opt.step(); sched.step()
        if ep % verbose_every == 0 or ep == 1:
            history.append({'ep': ep, 'loss': loss.item()})
            print(f'    ep {ep:5d}  loss={loss.item():.4e}  '
                  f'L1²={L1.pow(2).mean().item():.2e}  L2²={L2.pow(2).mean().item():.2e}  '
                  f'Le²={Le.pow(2).mean().item():.2e}  Ld²={Ld.pow(2).mean().item():.2e}  '
                  f'mc²={Lmc.pow(2).mean().item():.2e}')

    # L-BFGS polish (residual-weighted grid)
    print(f'  L-BFGS: max {n_lbfgs} outer iters, residual-weighted grid (8000 pts from 20000)')
    N_CAND = 20000; N_FROZEN = 8000
    with torch.no_grad():
        f_cand = sample_f(N_CAND).detach()
    f_cand_g = f_cand.clone().requires_grad_(True)
    L1c, L2c, Lec, Ldc, Lmcc = compute_residuals(net, f_cand_g)
    with torch.no_grad():
        res_total = (w_agent*(L1c.pow(2)+L2c.pow(2)) + w_asset*(Lec.pow(2)+Ldc.pow(2)) + w_mc*Lmcc.pow(2))
        weights = res_total.detach().abs().clamp(min=1e-16)
        weights = weights / weights.sum()
        idx = torch.multinomial(weights, N_FROZEN, replacement=False)
        f_grid = f_cand[idx].detach().requires_grad_(True)

    opt2 = optim.LBFGS(net.parameters(), lr=1.0, max_iter=30,
                       tolerance_grad=1e-16, tolerance_change=1e-18,
                       history_size=100, line_search_fn='strong_wolfe')
    prev_loss = float('inf')
    for outer in range(n_lbfgs):
        def closure():
            opt2.zero_grad()
            L1, L2, Le, Ld, Lmc = compute_residuals(net, f_grid)
            loss = (w_agent*(L1.pow(2).mean() + L2.pow(2).mean())
                  + w_asset*(Le.pow(2).mean() + Ld.pow(2).mean())
                  + w_mc   * Lmc.pow(2).mean())
            loss.backward(); return loss
        loss = opt2.step(closure).item()
        if outer % 10 == 0 or outer < 5:
            print(f'    outer {outer:3d}   loss={loss:.6e}')
        if abs(prev_loss - loss) < 1e-15:
            print(f'    stagnated at outer {outer}')
            break
        prev_loss = loss

    print(f'  Final L-BFGS loss: {loss:.6e}')
    return history, loss


def validate_run(net, compute_residuals, constants, n_plot=500,
                 title_suffix='', save_prefix=None, final_loss=None):
    '''Same 2×3 diagnostic plot as the MLP structural-baseline notebook.'''
    net.eval()
    g1, g2 = constants['gamma1'], constants['gamma2']
    is_homog = abs(g1 - g2) < 1e-12

    with torch.no_grad():
        f_bc = torch.tensor([0.0, 1.0], device=device)
        p1, p2, pe, pd = net(f_bc)
    print(f'\n  === Boundary check (γ¹={g1}, γ²={g2}) ===')
    print(f'    φ¹(1) = {p1[1].item():.6f}    target = {constants["PHI1_INF"]:.6f}    err = {abs(p1[1].item() - constants["PHI1_INF"]):.1e}')
    print(f'    φ²(0) = {p2[0].item():.6f}    target = {constants["PHI2_INF"]:.6f}    err = {abs(p2[0].item() - constants["PHI2_INF"]):.1e}')
    print(f'    φᵉ(0) = {pe[0].item():.6f}    target = {constants["PHIE_L"]:.6f}    err = {abs(pe[0].item() - constants["PHIE_L"]):.1e}')
    print(f'    φᵉ(1) = {pe[1].item():.6f}    target = {constants["PHIE_R"]:.6f}    err = {abs(pe[1].item() - constants["PHIE_R"]):.1e}')
    print(f'    φᵈ(0) = {pd[0].item():.6f}    target = {constants["PHID_L"]:.6f}    err = {abs(pd[0].item() - constants["PHID_L"]):.1e}')
    print(f'    φᵈ(1) = {pd[1].item():.6f}    target = {constants["PHID_R"]:.6f}    err = {abs(pd[1].item() - constants["PHID_R"]):.1e}')
    print(f'    φ¹(0) = {p1[0].item():.6f}    (off-side, learned)')
    print(f'    φ²(1) = {p2[1].item():.6f}    (off-side, learned)')

    f_plot = (F_EPS + (1-2*F_EPS) * torch.linspace(0, 1, n_plot, device=device)).requires_grad_(True)
    L1, L2, Le, Ld, Lmc = compute_residuals(net, f_plot)
    print(f'  === Residual summary (interior, N={n_plot}) ===')
    for name, L in [('L_φ¹', L1), ('L_φ²', L2), ('L_φᵉ', Le), ('L_φᵈ', Ld), ('L_mc', Lmc)]:
        arr = L.abs().detach().cpu().numpy()
        print(f'    {name}:  mean={arr.mean():.4e}    max={arr.max():.4e}')

    f_np = f_plot.detach().cpu().numpy()
    with torch.no_grad():
        phi1, phi2, phie, phid = net(f_plot.detach())
    phi1_np = phi1.cpu().numpy(); phi2_np = phi2.cpu().numpy()
    phie_np = phie.cpu().numpy(); phid_np = phid.cpu().numpy()
    mc_np = f_np*phi1_np + (1-f_np)*phi2_np - phie_np - phid_np

    # Reference for error panel
    if is_homog:
        phi1_ref = np.full_like(f_np, constants['PHI1_INF'])
        phi2_ref = np.full_like(f_np, constants['PHI2_INF'])
        phie_ref = np.full_like(f_np, constants['PHIE_L'])
        phid_ref = np.full_like(f_np, constants['PHID_L'])
        ref_panel_title = f'Error vs closed-form truth (γ={g1})'
        ref_ylabel = r'$|\widehat\phi - \phi_{\rm true}|$'
        print(f'  === Error vs truth (closed-form, homog γ={g1}) ===')
    else:
        with torch.enable_grad():
            f_ref = f_plot.detach().requires_grad_(True)
            phi1_ref = constants['phi1_base'](f_ref).detach().cpu().numpy()
            phi2_ref = constants['phi2_base'](f_ref).detach().cpu().numpy()
            phie_ref = constants['phie_base'](f_ref).detach().cpu().numpy()
            phid_ref = constants['phid_base'](f_ref).detach().cpu().numpy()
        ref_panel_title = f'Deviation from structural baseline (γ¹={g1}, γ²={g2})'
        ref_ylabel = r'$|\widehat\phi - \phi_{\rm base}|$'
        print(f'  === Deviation from structural baseline (no closed-form ref) ===')
    for name, phi, ref in [('φ¹', phi1_np, phi1_ref), ('φ²', phi2_np, phi2_ref),
                           ('φᵉ', phie_np, phie_ref), ('φᵈ', phid_np, phid_ref)]:
        err = np.abs(phi - ref)
        print(f'    {name}:  L∞ = {err.max():.4e}    L2 = {np.sqrt(np.mean(err**2)):.4e}')
    err1 = np.abs(phi1_np - phi1_ref); err2 = np.abs(phi2_np - phi2_ref)
    erre = np.abs(phie_np - phie_ref); errd = np.abs(phid_np - phid_ref)

    L1_abs = L1.abs().detach().cpu().numpy()
    L2_abs = L2.abs().detach().cpu().numpy()
    Le_abs = Le.abs().detach().cpu().numpy()
    Ld_abs = Ld.abs().detach().cpu().numpy()
    eps = 1e-16
    L1_rel = L1_abs / (np.abs(phi1_np) + eps)
    L2_rel = L2_abs / (np.abs(phi2_np) + eps)
    Le_rel = Le_abs / (np.abs(phie_np) + eps)
    Ld_rel = Ld_abs / (np.abs(phid_np) + eps)
    mc_scale = np.abs(f_np*phi1_np + (1-f_np)*phi2_np) + np.abs(phie_np + phid_np)
    mc_rel = np.abs(mc_np) / (0.5 * mc_scale + eps)

    print(f'  === Relative residual summary (|L_φ^i|/|φ^i|, interior) ===')
    for name, R in [('L_φ¹/φ¹', L1_rel), ('L_φ²/φ²', L2_rel),
                    ('L_φᵉ/φᵉ', Le_rel), ('L_φᵈ/φᵈ', Ld_rel),
                    ('L_mc/scale', mc_rel)]:
        print(f'    {name}:  mean={R.mean():.4e}    max={R.max():.4e}')

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))

    ax = axes[0,0]
    ax.plot(f_np, phi1_np, lw=2, color='steelblue', label=r'$\widehat\phi^1$')
    ax.plot(f_np, phi2_np, lw=2, color='firebrick', label=r'$\widehat\phi^2$')
    ax.scatter([1.0], [constants['PHI1_INF']], s=70, marker='o',
               facecolor='none', edgecolor='steelblue', lw=1.8, zorder=5,
               label=r'anchor $\phi^1_\infty$')
    ax.scatter([0.0], [constants['PHI2_INF']], s=70, marker='o',
               facecolor='none', edgecolor='firebrick', lw=1.8, zorder=5,
               label=r'anchor $\phi^2_\infty$')
    ax.set_xlabel('f'); ax.set_ylabel(r'$\phi^i(f)$'); ax.set_title('Agent WCR')
    ax.ticklabel_format(useOffset=False, style='plain', axis='y')
    ax.legend(fontsize=8, loc='best'); ax.grid(alpha=0.3)

    ax = axes[0,1]
    ax.plot(f_np, phie_np, lw=2, color='seagreen', label=r'$\widehat\phi^e$')
    ax.plot(f_np, phid_np, lw=2, color='darkorange', label=r'$\widehat\phi^d$')
    ax.plot(f_np, phie_np + phid_np, lw=1.5, ls='--', color='black', label=r'$\phi^e+\phi^d$')
    ax.scatter([0.0, 1.0], [constants['PHIE_L'], constants['PHIE_R']], s=60, marker='o',
               facecolor='none', edgecolor='seagreen', lw=1.8, zorder=5)
    ax.scatter([0.0, 1.0], [constants['PHID_L'], constants['PHID_R']], s=60, marker='o',
               facecolor='none', edgecolor='darkorange', lw=1.8, zorder=5)
    ax.set_xlabel('f'); ax.set_ylabel(r'$\phi(f)$'); ax.set_title('Asset prices')
    ax.ticklabel_format(useOffset=False, style='plain', axis='y')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax = axes[0,2]
    for name, err, col in [(r'$\phi^1$', err1, 'steelblue'),
                           (r'$\phi^2$', err2, 'firebrick'),
                           (r'$\phi^e$', erre, 'seagreen'),
                           (r'$\phi^d$', errd, 'darkorange')]:
        ax.semilogy(f_np, err + 1e-16, lw=1.2, label=name, color=col)
    ax.set_xlabel('f'); ax.set_ylabel(ref_ylabel)
    ax.set_title(ref_panel_title); ax.legend(fontsize=9); ax.grid(alpha=0.3, which='both')

    ax = axes[1,0]
    for name, L_abs, col in [(r'$L_{\phi^1}$', L1_abs, 'steelblue'),
                             (r'$L_{\phi^2}$', L2_abs, 'firebrick'),
                             (r'$L_{\phi^e}$', Le_abs, 'seagreen'),
                             (r'$L_{\phi^d}$', Ld_abs, 'darkorange')]:
        ax.semilogy(f_np, L_abs + 1e-16, lw=1.2, label=name, color=col)
    ax.set_xlabel('f'); ax.set_ylabel('|L|'); ax.set_title('ODE residuals — absolute (log)')
    ax.legend(fontsize=9); ax.grid(alpha=0.3, which='both')

    ax = axes[1,1]
    for name, R, col in [(r'$|L_{\phi^1}|/\phi^1$', L1_rel, 'steelblue'),
                         (r'$|L_{\phi^2}|/\phi^2$', L2_rel, 'firebrick'),
                         (r'$|L_{\phi^e}|/\phi^e$', Le_rel, 'seagreen'),
                         (r'$|L_{\phi^d}|/\phi^d$', Ld_rel, 'darkorange')]:
        ax.semilogy(f_np, R + 1e-16, lw=1.2, label=name, color=col)
    ax.set_xlabel('f'); ax.set_ylabel('|L|/|φ|'); ax.set_title('ODE residuals — relative (log)')
    ax.legend(fontsize=9); ax.grid(alpha=0.3, which='both')

    ax = axes[1,2]
    ax.semilogy(f_np, mc_rel + 1e-16, lw=1.5, color='purple')
    ax.set_xlabel('f'); ax.set_ylabel(r'$|L_{mc}|/\frac{1}{2}(|\sum f\phi^i|+|\phi^e+\phi^d|)$')
    ax.set_title('Market clearing — relative (log)')
    ax.grid(alpha=0.3, which='both')

    suptxt = rf'Spectral PINN K={constants["K"]} ($\gamma^1={g1}$, $\gamma^2={g2}$){title_suffix}'
    if final_loss is not None:
        suptxt += f'\n(final loss = {final_loss:.2e})'
    plt.suptitle(suptxt, fontsize=14, fontweight='bold')
    plt.tight_layout()
    if save_prefix:
        plt.savefig(f'{save_prefix}.png', dpi=140, bbox_inches='tight')
    plt.show()

print('train_run and validate_run (spectral) defined.')


---
## 3. Run 1 — Homog log utility (γ¹ = γ² = 1)

In the homog limit the true solution is a constant in f, so all
Chebyshev coefficients should converge near zero (the structural
baseline already gives the constant exactly). Expect machine-precision
loss.


In [ ]:
print('=' * 70)
print('RUN 1: γ¹ = γ² = 1 (homog log) — spectral')
print('=' * 70)

net1, compute_residuals1, constants1 = setup_run(gamma1=1.0, gamma2=1.0, K=15)
print(f'  Anchors: φ={constants1["PHI1_INF"]:.4f}, φᵉ={constants1["PHIE_L"]:.4f}, φᵈ={constants1["PHID_L"]:.4f}')

history1, final_loss1 = train_run(net1, compute_residuals1, n_adam=1500, n_lbfgs=100)
validate_run(net1, compute_residuals1, constants1, title_suffix=' — RUN 1',
             save_prefix='run1_spectral_log', final_loss=final_loss1)


---
## 4. Run 2 — Homog CRRA (γ¹ = γ² = 2)


In [ ]:
print('=' * 70)
print('RUN 2: γ¹ = γ² = 2 (homog CRRA) — spectral')
print('=' * 70)

net2, compute_residuals2, constants2 = setup_run(gamma1=2.0, gamma2=2.0, K=15)
print(f'  Anchors: φ={constants2["PHI1_INF"]:.4f}, φᵉ={constants2["PHIE_L"]:.4f}, φᵈ={constants2["PHID_L"]:.4f}')

history2, final_loss2 = train_run(net2, compute_residuals2, n_adam=1500, n_lbfgs=100)
validate_run(net2, compute_residuals2, constants2, title_suffix=' — RUN 2',
             save_prefix='run2_spectral_crra', final_loss=final_loss2)


---
## 5. Run 3 — True heterogeneous (γ¹ = 1, γ² = 2) — PRODUCTION

The real test. With K=15 we have 64 trainable coefficients total.
If spectral helps, we should see:
- Lower final loss than the MLP version (~10⁻⁴ → ?)
- Cleaner residual profiles (no MLP-specific wiggles)
- Faster L-BFGS convergence (better-conditioned Hessian)


In [ ]:
print('=' * 70)
print('RUN 3: γ¹ = 1, γ² = 2 (true heterogeneous, production) — spectral')
print('=' * 70)

net3, compute_residuals3, constants3 = setup_run(gamma1=1.0, gamma2=2.0, K=15)
print(f'  Anchors: φ¹(1)={constants3["PHI1_INF"]:.4f}, φ²(0)={constants3["PHI2_INF"]:.4f}')
print(f'           φᵉ(0)={constants3["PHIE_L"]:.4f}, φᵉ(1)={constants3["PHIE_R"]:.4f}')
print(f'           φᵈ(0)={constants3["PHID_L"]:.4f}, φᵈ(1)={constants3["PHID_R"]:.4f}')

history3, final_loss3 = train_run(net3, compute_residuals3, n_adam=2000, n_lbfgs=150)
validate_run(net3, compute_residuals3, constants3, title_suffix=' — RUN 3 (production)',
             save_prefix='run3_spectral_het', final_loss=final_loss3)


---
## 6. Summary

Compare the final losses to the MLP structural-baseline notebook to see
whether spectral parameterization helps:
- MLP version (structural baseline): Run 3 final loss ≈ 2.16e-04
- Spectral version (K=15): Run 3 final loss = (see below)

If the spectral version reaches a substantially lower floor with
fewer parameters, it confirms that the MLP was overparameterized for
this smooth 1D problem.

Inspect the learned coefficients of `net3.N2` to see the spectral
decomposition of the heterogeneous correction.


In [ ]:
print('=' * 70)
print('SUMMARY OF ALL RUNS — Spectral PINN (K=15)')
print('=' * 70)
print(f'  Run 1 (γ¹=γ²=1, homog log):  final loss = {final_loss1:.6e}')
print(f'  Run 2 (γ¹=γ²=2, homog CRRA): final loss = {final_loss2:.6e}')
print(f'  Run 3 (γ¹=1, γ²=2, het):     final loss = {final_loss3:.6e}')

print('\n=== Learned Chebyshev coefficients for Run 3 (production) ===')
for name, mod in [('N1 (log agent)', net3.N1), ('N2 (CRRA-2 agent)', net3.N2),
                  ('Ne (equity)', net3.Ne), ('Nd (dividend)', net3.Nd)]:
    coeffs = mod.coeffs.detach().cpu().numpy()
    print(f'  {name}:  {coeffs[:8]}...  (max |a_k| = {np.abs(coeffs).max():.3e})')

torch.save(net3.state_dict(), 'net_het_spectral_g1_g2.pt')
print(f'\nSaved production network -> net_het_spectral_g1_g2.pt')

# Optional: spectrum plot
fig, ax = plt.subplots(figsize=(8, 4))
for name, mod, col in [('N¹', net3.N1, 'steelblue'), ('N²', net3.N2, 'firebrick'),
                       ('Nᵉ', net3.Ne, 'seagreen'), ('Nᵈ', net3.Nd, 'darkorange')]:
    coeffs = np.abs(mod.coeffs.detach().cpu().numpy())
    ax.semilogy(np.arange(len(coeffs)), coeffs + 1e-20, 'o-', lw=1.5, color=col, label=name)
ax.set_xlabel('Chebyshev mode k'); ax.set_ylabel('|a_k|')
ax.set_title('Learned Chebyshev spectrum (Run 3 — production)')
ax.legend(fontsize=10); ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('spectral_coefficients.png', dpi=140, bbox_inches='tight')
plt.show()
